In [49]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import KNNImputer

In [74]:
df_train = pd.read_csv('train.csv')
df_test = pd.read_csv('test.csv')
df_test['Transported'] = False
df = pd.concat([df_train , df_test] , sort = False)
df.drop(['Name' ,'PassengerId'] , axis = 1  , inplace = True)
df.head()

,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Transported
0,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,False
1,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,True
2,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,False
3,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,False
4,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,True


In [75]:
df.shape[0] == df_train.shape[0] + df_test.shape[0]

True

In [76]:
df.isna().sum()

HomePlanet      288
CryoSleep       310
Cabin           299
Destination     274
Age             270
VIP             296
RoomService     263
FoodCourt       289
ShoppingMall    306
Spa             284
VRDeck          268
Transported       0
dtype: int64

In [77]:
# first chagneing the format of Cabin deck/num/side

df[['Deck' , 'Num' , 'Side']] = df['Cabin'].str.split('/' , expand= True)
df = df.drop(columns = ['Cabin'])
df['Deck'] = df['Deck'].fillna('U')
df['Num'] = df['Num'].fillna(-1)
df['Side'] =  df['Side'].fillna('U')

In [78]:
df.head()

,HomePlanet,CryoSleep,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Transported,Deck,Num,Side
0,Europa,False,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,False,B,0,P
1,Earth,False,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,True,F,0,S
2,Europa,False,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,False,A,0,S
3,Europa,False,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,False,A,0,S
4,Earth,False,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,True,F,1,S


In [79]:
df.isna().sum()

HomePlanet      288
CryoSleep       310
Destination     274
Age             270
VIP             296
RoomService     263
FoodCourt       289
ShoppingMall    306
Spa             284
VRDeck          268
Transported       0
Deck              0
Num               0
Side              0
dtype: int64

In [80]:
df['Side'].value_counts()

Side
S    6381
P    6290
U     299
Name: count, dtype: int64

In [81]:
df['Deck'] = df['Deck'].map({'G' : 0 , 'F' : 1 , 'E' : 2 , 'D' : 3 , 'C' : 4 , 'B' :5 , 'A' : 6 , 'U' : 7 , 'T' : 8})
df['Side'] = df['Side'].map({'U' : -1 , 'P' : 1 , 'S' : 2})

In [82]:
impute_lis = ['Age' , 'CryoSleep' , 'VIP' , 'RoomService' ,	'FoodCourt' ,	'ShoppingMall' ,	'Spa',	'VRDeck'	,'Transported'	,'Deck',	'Num',	'Side']
rest = list(set(df.columns) - set(impute_lis))
df_rest = df[rest]
imp = KNNImputer()
df_imputed = imp.fit_transform(df[impute_lis])
df_imputed = pd.DataFrame(df_imputed , columns = impute_lis)
df = pd.concat([df_rest.reset_index(drop = True) , df_imputed.reset_index(drop = True)] , axis = 1)

In [83]:
df['HomePlanet'] = df['HomePlanet'].fillna('U')
df['Destination'] = df['Destination'].fillna('U')
category_colls = ['HomePlanet' , 'Destination']

for col in category_colls:
    df = pd.concat([df , pd.get_dummies(df[col] , prefix = col)] , axis = 1)

df = df.drop(columns = category_colls)

In [84]:
df.head()

,Age,CryoSleep,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Transported,Deck,Num,Side,HomePlanet_Earth,HomePlanet_Europa,HomePlanet_Mars,HomePlanet_U,Destination_55 Cancri e,Destination_PSO J318.5-22,Destination_TRAPPIST-1e,Destination_U
0,39.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.0,0.0,1.0,False,True,False,False,False,False,True,False
1,24.0,0.0,0.0,109.0,9.0,25.0,549.0,44.0,1.0,1.0,0.0,2.0,True,False,False,False,False,False,True,False
2,58.0,0.0,1.0,43.0,3576.0,0.0,6715.0,49.0,0.0,6.0,0.0,2.0,False,True,False,False,False,False,True,False
3,33.0,0.0,0.0,0.0,1283.0,371.0,3329.0,193.0,0.0,6.0,0.0,2.0,False,True,False,False,False,False,True,False
4,16.0,0.0,0.0,303.0,70.0,151.0,565.0,2.0,1.0,1.0,1.0,2.0,True,False,False,False,False,False,True,False


In [85]:
bill_cols = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
df['amt_spent'] = df[bill_cols].sum(axis = 1)
df['std_amt_spent'] = df[bill_cols].std(axis = 1)
df['mean_amt_spent'] = df[bill_cols].mean(axis = 1)


In [86]:
df.corr()

,Age,CryoSleep,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Transported,Deck,...,HomePlanet_Europa,HomePlanet_Mars,HomePlanet_U,Destination_55 Cancri e,Destination_PSO J318.5-22,Destination_TRAPPIST-1e,Destination_U,amt_spent,std_amt_spent,mean_amt_spent
Age,1.000000,-0.066347,0.081179,0.067359,0.122581,0.036350,0.117298,0.101997,-0.050223,0.224153,...,0.218538,0.024158,0.002542,0.021968,-0.032192,-0.000549,0.004735,0.180209,0.175623,0.180209
CryoSleep,-0.066347,1.000000,-0.081132,-0.257334,-0.212538,-0.222773,-0.204353,-0.195215,0.324411,-0.008170,...,0.102179,0.033463,0.000293,0.069187,0.087516,-0.108606,-0.017894,-0.385675,-0.387950,-0.385675
VIP,0.081179,-0.081132,1.000000,0.061440,0.122624,0.025383,0.080553,0.111077,-0.018720,0.156045,...,0.139783,0.045486,-0.000597,0.038727,-0.006223,-0.026614,-0.009939,0.165779,0.154094,0.165779
RoomService,0.067359,-0.257334,0.061440,1.000000,-0.018742,0.059918,0.010314,-0.023413,-0.174833,0.031005,...,-0.073868,0.253273,-0.004947,-0.023413,-0.061034,0.059823,-0.005605,0.224355,0.218361,0.224355
FoodCourt,0.122581,-0.212538,0.122624,-0.018742,1.000000,0.000593,0.229477,0.242398,0.034737,0.285974,...,0.363065,-0.127267,-0.012598,0.130852,-0.062081,-0.071576,-0.010675,0.745149,0.751900,0.745149
ShoppingMall,0.036350,-0.222773,0.025383,0.059918,0.000593,1.000000,0.013030,0.003790,0.004171,0.020581,...,-0.032314,0.125037,0.001113,-0.015714,-0.029697,0.032645,-0.002061,0.228318,0.215404,0.228318
Spa,0.117298,-0.204353,0.080553,0.010314,0.229477,0.013030,1.000000,0.147174,-0.154815,0.221104,...,0.268730,-0.090498,-0.005441,0.086701,-0.052796,-0.042921,0.001723,0.591452,0.540164,0.591452
VRDeck,0.101997,-0.195215,0.111077,-0.023413,0.242398,0.003790,0.147174,1.000000,-0.142771,0.223607,...,0.276977,-0.110336,0.005947,0.087479,-0.042784,-0.046709,-0.008262,0.604440,0.566090,0.604440
Transported,-0.050223,0.324411,-0.018720,-0.174833,0.034737,0.004171,-0.154815,-0.142771,1.000000,0.077959,...,0.131977,0.005643,0.006403,0.083625,0.000760,-0.072731,-0.000554,-0.140452,-0.121171,-0.140452
Deck,0.224153,-0.008170,0.156045,0.031005,0.285974,0.020581,0.221104,0.223607,0.077959,1.000000,...,0.775551,-0.050491,-0.006267,0.244529,-0.182194,-0.096539,-0.007727,0.351490,0.336630,0.351490
